In [1]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, iterable))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# # num_nodes = 4
# zone_no = 0

# # Use 2 extra 2-core machines as client machines.
# n_clients = 2

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = CLIENT_DURATION_SEC + 45
# CLIENT_TOTAL_REQUESTS = 100000000
# CLIENT_MAX_IN_FLIGHT = 400
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Throughput/latency load points.
# # This gives enough points for a throughput-vs-latency plot without too many subruns.
# LOAD_POINTS = [
#     # {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},   # total inflight 100
#     {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},   # total inflight 150
#     # {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},   # total inflight 200
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
#     # {"active_clients": 1, "client_threads": 4, "max_in_flight": 100},   # total inflight 400
#     # {"active_clients": 1, "client_threads": 8, "max_in_flight": 100},   # total inflight 800
#     # {"active_clients": 2, "client_threads": 8, "max_in_flight": 100},   # total inflight 1600
# ]


# for num_nodes in [8]:
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     def get_zone_for_instance(i):
#         if i < int(num_nodes / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     # Fetch all tsm-sc-* instances across ALL zones
#     fetch_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     '''

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     print("\n➡ Existing instances to delete:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     def delete_instance(instance):
#         name, inst_zone = instance
#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     # if instances:
#     #     run_parallel(delete_instance, instances, max_workers=32)
#     #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
#     # else:
#     #     print("\n✔ No tsm-sc-* instances found.\n")

#     # Create commands list
#     commands = []

#     # Create replica nodes.
#     for i in range(num_nodes):
#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{i:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     # Create client machines after replica nodes.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{client_idx:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     # run_parallel(run_command, commands, max_workers=48)

#     print("All instances launched.")

#     # Get sorted node and client IPs.
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#     n_collection = 100
#     subprocess.call('make -j8', shell=True)

#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     results = run_parallel(
#         kill_stellar_private,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )

#     def git_pull_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     results = run_parallel(
#         git_pull_stellar,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )
#     print(results)

#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     target_file = "../stellar-private/node2/stellar-core.cfg"
#     line_to_add = "MEMORY_PROF=true"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     def compile_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     # results = run_parallel(
#     #     compile_stellar,
#     #     range(num_nodes + n_clients),
#     #     max_workers=48
#     # )
#     print(results)

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # No sleep intervals. SEND_INTERVAL_US is fixed at 0.
#     # We vary client concurrency to create the throughput-vs-latency points.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]
#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight
        
#         total_client_threads = active_clients * client_threads

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=48
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(60)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"shab_tput_latency_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={CLIENT_MAX_IN_FLIGHT}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, num_nodes)),
#             max_workers=48
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
from pathlib import Path


def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


# latencies: 50, 90 150, 210
default_region = ['us-central1-c']
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# Regions

zone_no = 0

# Run throughput/latency vs num_nodes for these system sizes.
NUM_NODES_LIST = [4, 8, 16, 32, 48]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Delete all tsm-sc-* instances before each num_nodes run.
DELETE_BEFORE_EACH_RUN = True

# Also delete all tsm-sc-* instances after each num_nodes run to reduce cost.
DELETE_AFTER_EACH_RUN = True

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 200
]


for num_nodes in NUM_NODES_LIST:
# for zone_no in  [0,1,2,3, 4]:

    project = "research-488322"
    zone = "us-central1-c"
    machine_type = "e2-standard-2"
    image_family = "tsm-sc-family"  # your custom image
    subnet = "default"
    gcp_username = "tejas"

    print("\n" + "#" * 100)
    print(f"Starting experiment for num_nodes={num_nodes}")
    print("#" * 100)

    def get_zone_for_instance(i):
        if i < int(num_nodes / 2):
            return default_region[0]
        else:
            return regions[zone_no]

    def fetch_existing_instances():
        fetch_cmd = f'''
        gcloud compute instances list \
            --project={project} \
            --filter="name~'^tsm-sc-'" \
            --format="value(name,zone)"
        '''

        output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
        instances = []

        for line in output.splitlines():
            if line.strip():
                name, inst_zone = line.split()
                instances.append((name, inst_zone))

        return instances

    def delete_instance(instance):
        name, inst_zone = instance

        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={inst_zone} \
            --project={project} \
            --quiet
        '''

        print(f"🗑️ Deleting {name} in {inst_zone}")
        return subprocess.call(cmd, shell=True)

    # -------------------------------------------------------------------------
    # Delete existing tsm-sc-* instances before this num_nodes run.
    # This keeps cost low and avoids stale machines/configs from prior runs.
    # -------------------------------------------------------------------------
    if DELETE_BEFORE_EACH_RUN:
        instances = fetch_existing_instances()

        print("\n➡ Existing instances to delete before run:")
        for name, inst_zone in instances:
            print(f"  - {name} ({inst_zone})")

        if instances:
            run_parallel(
                delete_instance,
                instances,
                max_workers=min(32, len(instances))
            )
            print("\n🧹 All existing tsm-sc-* instances deleted.\n")
        else:
            print("\n✔ No existing tsm-sc-* instances found.\n")

    # -------------------------------------------------------------------------
    # Create replica and client machines for this num_nodes run.
    # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
    # Clients:  tsm-sc-num_nodes ...
    # -------------------------------------------------------------------------
    commands = []

    # Create replica nodes.
    for i in range(num_nodes):
        inst_zone = get_zone_for_instance(i)

        cmd = f'''
        gcloud compute instances create tsm-sc-{i:03} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())

    # Create client machines after replica nodes.
    for i in range(n_clients):
        client_idx = num_nodes + i
        inst_zone = get_zone_for_instance(client_idx)

        cmd = f'''
        gcloud compute instances create tsm-sc-{client_idx:03} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())

    def run_command(command):
        print(f"Running: {command}")
        return subprocess.call(command, shell=True)

    run_parallel(
        run_command,
        commands,
        max_workers=min(48, len(commands))
    )

    print("All instances launched.")

    # Give GCP/SSH a little time after VM creation.
    time.sleep(30)

    # -------------------------------------------------------------------------
    # Get sorted node and client IPs.
    # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
    # Client IPs must not be included.
    # -------------------------------------------------------------------------
    ip_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --sort-by=name \
        --format="value(name,zone,networkInterfaces[0].networkIP)"
    '''

    ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

    instance_records = []

    for line in ip_output.splitlines():
        if line.strip():
            name, inst_zone, ip = line.split()
            idx = int(name.rsplit("-", 1)[1])
            instance_records.append((idx, name, inst_zone, ip))

    instance_records.sort()

    node_records = [r for r in instance_records if r[0] < num_nodes]
    client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

    if len(node_records) != num_nodes:
        raise RuntimeError(
            f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
        )

    if len(client_records) != n_clients:
        raise RuntimeError(
            f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
        )

    iplist = [r[3] for r in node_records]

    with open("tsm_ips.txt", "w") as f:
        for ip in iplist:
            f.write(ip + "\n")

    print("🎯 Node IPs:", iplist)
    print("🎯 Client instances:", client_records)

    node1_ip = iplist[0]
    print(f"Clients will connect to leader/node1 at: {node1_ip}")

    # Push latest code/config changes.
    subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

    n_collection = 100
    subprocess.call('make -j8', shell=True)

    def kill_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = f"""\
cd /home/tejas/stellar-private; \
sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    results = run_parallel(
        kill_stellar_private,
        range(num_nodes + n_clients),
        max_workers=min(48, num_nodes + n_clients)
    )

    def git_pull_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    results = run_parallel(
        git_pull_stellar,
        range(num_nodes + n_clients),
        max_workers=min(48, num_nodes + n_clients)
    )
    print(results)

    # -------------------------------------------------------------------------
    # Compile on every created machine for this num_nodes run.
    # -------------------------------------------------------------------------
    def compile_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j16; \
cd; \
sudo rm -rf stellar-private"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    results = run_parallel(
        compile_stellar,
        range(num_nodes + n_clients),
        max_workers=min(48, num_nodes + n_clients)
    )
    print(results)

    # -------------------------------------------------------------------------
    # Generate stellar-private configs locally using only replica IPs.
    # -------------------------------------------------------------------------
    stellar_private_path = Path('../stellar-private')
    if stellar_private_path.exists():
        shutil.rmtree(stellar_private_path)
    stellar_private_path.mkdir()

    subprocess.call(
        'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
        shell=True
    )

    subprocess.call(
        'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
        './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
        shell=True
    )

    # Enable custom message only on leader.
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg"

    subprocess.call(
        f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
        shell=True
    )

    # Optional memory profiling on node2.
    # if num_nodes >= 2:
    #     target_file = "../stellar-private/node2/stellar-core.cfg"
    #     line_to_add = "MEMORY_PROF=true"

    #     subprocess.call(
    #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
    #         shell=True
    #     )

    #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

    # -------------------------------------------------------------------------
    # Throughput/latency experiment loop.
    # Fixed offered load for scalability:
    # active_clients=1, client_threads=2, max_in_flight=100.
    # Aggregate max in-flight = 200.
    # -------------------------------------------------------------------------
    for load in LOAD_POINTS:
        active_clients = load["active_clients"]
        client_threads = load["client_threads"]
        client_max_in_flight = load["max_in_flight"]

        if active_clients > n_clients:
            raise RuntimeError(
                f"active_clients={active_clients} exceeds n_clients={n_clients}"
            )

        total_client_threads = active_clients * client_threads
        aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

        run_label = (
            f"clients_{active_clients}_threads_{client_threads}_"
            f"inflight_{client_max_in_flight}_"
            f"total_threads_{total_client_threads}_"
            f"total_inflight_{aggregate_max_in_flight}"
        )

        print("\n" + "=" * 80)
        print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
        print("=" * 80)

        def clean_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = f"""\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        results = run_parallel(
            clean_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        def copy_folder_to_instance(
            i,
            source_folder="/home/tejas/stellar-private",
            destination_path="/home/tejas/stellar-private"
        ):
            inst_zone = get_zone_for_instance(i)
            instance_name = f"tsm-sc-{i:03}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

            print(f"Executing command for {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Command for {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        results = run_parallel(
            copy_folder_to_instance,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        def run_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            node_number = i + 1
            instance_name = f"tsm-sc-{i:03}"

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # Kill old processes on all replica and client machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        # Start consensus replicas.
        results = run_parallel(
            run_stellar_private,
            range(num_nodes),
            max_workers=min(48, num_nodes)
        )

        print(results)
        print("All Stellar nodes should be starting in the background.")

        # Give nodes time to authenticate and start the client listener.
        time.sleep(60)

        def run_stellar_client(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing client command: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # Start only the required number of client VMs for this load point.
        active_client_indices = [
            num_nodes + j for j in range(active_clients)
        ]

        results = run_parallel(
            run_stellar_client,
            active_client_indices,
            max_workers=active_clients
        )

        print(results)
        print(
            f"Started {active_clients} client VM(s), "
            f"each with {client_threads} client threads. "
            f"Total client threads = {total_client_threads}. "
            f"Aggregate max in-flight = {aggregate_max_in_flight}."
        )

        # Wait for the duration run to produce stable per-second client logs.
        # The clients may wait forever on final partial batches, so we kill them after this.
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)

        # Stop all nodes and clients.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients)
        )

        remote_base_folder = "/home/tejas/stellar-private"

        local_base_destination = (
            "/home/tejas/work/experiments/shabdiz/"
            + f"shab_vs_num_nodes_{num_nodes}_{run_label}"
        )

        Path(local_base_destination).mkdir(parents=True, exist_ok=True)

        # Save run metadata.
        with open(Path(local_base_destination) / "run_config.txt", "w") as f:
            f.write(f"num_nodes={num_nodes}\n")
            f.write(f"active_clients={active_clients}\n")
            f.write(f"client_threads_per_vm={client_threads}\n")
            f.write(f"total_client_threads={total_client_threads}\n")
            f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
            f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
            f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
            f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
            f.write(f"leader_ip={node1_ip}\n")
            f.write(f"machine_type={machine_type}\n")
            f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

        def copy_folder_from_instance(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"

            node_number = i + 1
            node_folder = f"node{node_number}"

            remote_source_path = posixpath.join(remote_base_folder, node_folder)

            local_destination_path = Path(local_base_destination) / instance_name
            local_destination_path.mkdir(parents=True, exist_ok=True)

            remote_source = f"{instance_name}:{remote_source_path}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

            print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Copy from {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        # Copy only a few node logs to reduce time.
        # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
        node_copy_results = run_parallel(
            copy_folder_from_instance,
            range(min(3, num_nodes)),
            max_workers=min(48, max(1, min(3, num_nodes)))
        )

        def copy_client_log(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_source = (
                f"{instance_name}:/home/tejas/stellar-private/"
                f"stellar-client-{client_id}.log"
            )

            local_destination_path = Path(local_base_destination)
            local_destination_path.mkdir(parents=True, exist_ok=True)

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

            print(f"Copying client log from {instance_name}...")

            output = subprocess.call(command, shell=True)

            print(f"Copy finished with exit code: {output}")

            return (instance_name, output)

        client_copy_results = run_parallel(
            copy_client_log,
            active_client_indices,
            max_workers=active_clients
        )

        print("\n--- Summary of Download Results ---")
        print("Node log copies:", node_copy_results)
        print("Client log copies:", client_copy_results)
        print(f"Saved run to: {local_base_destination}")

    # -------------------------------------------------------------------------
    # Delete instances after this num_nodes run to reduce cost.
    # This is especially useful after the final 48-node run.
    # -------------------------------------------------------------------------
    if DELETE_AFTER_EACH_RUN:
        instances = fetch_existing_instances()

        print("\n➡ Existing instances to delete after run:")
        for name, inst_zone in instances:
            print(f"  - {name} ({inst_zone})")

        if instances:
            run_parallel(
                delete_instance,
                instances,
                max_workers=min(32, len(instances))
            )
            print(f"\n🧹 Deleted all tsm-sc-* instances after num_nodes={num_nodes}.\n")
        else:
            print("\n✔ No tsm-sc-* instances found after run.\n")


####################################################################################################
Starting experiment for num_nodes=4
####################################################################################################

➡ Existing instances to delete before run:
  - tsm-sc-001 (us-central1-c)
  - tsm-sc-013 (us-central1-c)
  - tsm-sc-019 (us-central1-c)
  - tsm-sc-021 (us-central1-c)
  - tsm-sc-025 (us-central1-c)
  - tsm-sc-026 (us-central1-c)
  - tsm-sc-032 (us-central1-c)
🗑️ Deleting tsm-sc-001 in us-central1-c
🗑️ Deleting tsm-sc-013 in us-central1-c
🗑️ Deleting tsm-sc-019 in us-central1-c
🗑️ Deleting tsm-sc-021 in us-central1-c
🗑️ Deleting tsm-sc-025 in us-central1-c
🗑️ Deleting tsm-sc-026 in us-central1-c
🗑️ Deleting tsm-sc-032 in us-central1-c


ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-026' was not found

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-021' was not found

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-019' was not found

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-025' was not found

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-013' was not found

ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/research-488322/zones/us-central1-c/instances/tsm-sc-001' was not found

Dele


🧹 All existing tsm-sc-* instances deleted.

Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-c             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-003].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-003  us-central1-c  e2-standard-2               10.128.0.43  34.57.205.242  RUNNING


...

NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP    EXTERNAL_IP  STATUS
tsm-sc-007  us-central1-c  e2-standard-2               10.128.15.205  35.239.37.2  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-008].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-008  us-central1-c  e2-standard-2               10.128.0.3   34.69.218.105  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-c/instances/tsm-sc-048].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP    EXTERNAL_IP    STATUS
tsm-sc-048  us-central1-c  e2-standard-2               10.128.15.214  34.55.151.187  RUNNING
All instances launched.


In [ ]:

    # # Fetch all tsm-sc-* instances across ALL zones
    # fetch_cmd = f'''
    # gcloud compute instances list \
    #     --project={project} \
    #     --filter="name~'^tsm-sc-'" \
    #     --format="value(name,zone)"
    # '''
    
    # output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    # instances = []
    
    # for line in output.splitlines():
    #     if line.strip():
    #         name, zone = line.split()
    #         instances.append((name, zone))
    
    # print("\n➡ Existing instances to delete:")
    # for name, zone in instances:
    #     print(f"  - {name} ({zone})")
    
    # def delete_instance(name, zone):
    #     cmd = f'''
    #     gcloud compute instances delete {name} \
    #         --zone={zone} \
    #         --project={project} \
    #         --quiet
    #     '''
    #     print(f"🗑️ Deleting {name} in {zone}")
    #     return subprocess.call(cmd, shell=True)
    
    # if instances:
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    #         futures = [
    #             executor.submit(delete_instance, name, zone)
    #             for name, zone in instances
    #         ]
    #         concurrent.futures.wait(futures)
    
    #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    # else:
    #     print("\n✔ No tsm-sc-* instances found.\n")